# DragonHPC Multiprocessing Tutorial

**Estimated time:** ~30-40 minutes  
**Format:** 5 short exercises - each has a problem and a hidden answer you can reveal.

---

## What you will learn

This notebook introduces Python multiprocessing with DragonHPC for single-node
and multi-node runs. You will practice how to:

- launch processes
- synchronize worker execution
- communicate data between processes
- observe process placement across nodes

The exercises are beginner friendly, while also connecting to concepts that
intermediate and advanced users already know from Python multiprocessing.

---

## Setup - run this first!

Start Jupyter from a Dragon-enabled Python environment (for example, with
`dragon-jupyter`) and then run the next cell.

In [ ]:
import os
import socket
import time
import dragon
import multiprocessing as mp

# Use Dragon as the multiprocessing backend.
mp.set_start_method("dragon")

print("Dragon multiprocessing ready.")

`multiprocessing` synchronization and communication objects (for example
`Queue`, `Event`, `Lock`, `Value`) happily work with `"dragon"` set as
the start method in `multiprocessing` just as they would with any of the
other start methods in `multiprocessing` (i.e. `"spawn"`, `"fork"`, or
`"forkserver"`).

When the Dragon runtime is available, a number of opportunities for
optimization arise which is why there are additional tools available
in the `dragon` module.  After exercising the use of Dragon through
standard `multiprocessing`, we will begin by introducing the Dragon
Native `Process` object which exposes further control over the creation
and placement of processes, noting how it intentionally mirrors the
familiar multiprocessing API patterns.

---

## Exercise 1 - Parallel map with multiprocessing Pool

**Background:**

`multiprocessing.Pool` is useful when the same function should run over many
inputs. With Dragon configured as the multiprocessing backend, you can use
`Pool.map` for data-parallel workloads.

Ref: https://docs.python.org/3/library/multiprocessing.html#introduction

Demo pattern:

```python
def double(x):
    return 2 * x

with mp.Pool(processes=4) as pool:
    out = pool.map(double, [1, 2, 3, 4])

print(out)  # [2, 4, 6, 8]
```

**Your task:**

1. Write a function `transform(x)` that returns `x * x + 1`
2. Use `mp.Pool(processes=4)`
3. Call `pool.map(transform, data)` where `data = list(range(12))`
4. Print the mapped results
5. Print the sum of the mapped results

In [ ]:
# -- Exercise 1 -- your code here -----------------------------------------------


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def transform(x):
    return x * x + 1

data = list(range(12))

with mp.Pool(processes=4) as pool:
    mapped = pool.map(transform, data)

print(mapped)
print("sum:", sum(mapped))
```

Expected mapped output:
```
[1, 2, 5, 10, 17, 26, 37, 50, 65, 82, 101, 122]
```

</details>

---

## Exercise 2 - Launch processes and join

**Background:**

A process is started with `start()` and waited on with `join()`. This is the
first building block for parallel programs.

Here is the core Dragon Native process pattern:

```python
def hello_worker(name):
    print(f"hello from {name}, pid={os.getpid()}")

p = dragon.native.Process(target=hello_worker, args=("dragon",))
p.start()
p.join()
```

**Your task:**

1. Write a worker function `square_worker(i)` that prints the worker id, PID, and `i*i`
2. Launch 4 processes using `dragon.native.Process`
3. Start and join all processes
4. Print `All workers finished` at the end

In [ ]:
# -- Exercise 2 -- your code here -----------------------------------------------


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def square_worker(i):
    print(f"worker={i} pid={os.getpid()} square={i*i}")

procs = [dragon.native.Process(target=square_worker, args=(i,)) for i in range(4)]

for p in procs:
    p.start()

for p in procs:
    p.join()

print("All workers finished")
```

</details>

---

## Exercise 3 - Data communication with Queue

**Background:**

`mp.Queue` provides process-safe message passing. Producers can put data and
a consumer (or parent process) can read and aggregate results.

Demo pattern:

```python
q = mp.Queue()

def say_hello(q):
    q.put(("msg", "hello from worker"))

p = dragon.native.Process(target=say_hello, args=(q,))
p.start()
message = q.get()
p.join()
print(message)
```

**Your task:**

1. Write `producer(worker_id, out_q)` that sends `(worker_id, worker_id**3)`
2. Launch 6 producers
3. Read 6 messages from the queue in the parent process
4. Store results in a dictionary and print it sorted by key

In [ ]:
# -- Exercise 3 -- your code here -----------------------------------------------


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def producer(worker_id, out_q):
    out_q.put((worker_id, worker_id**3))

out_q = mp.Queue()
procs = [dragon.native.Process(target=producer, args=(i, out_q)) for i in range(6)]

for p in procs:
    p.start()

results = {}
for _ in range(6):
    worker_id, value = out_q.get()
    results[worker_id] = value

for p in procs:
    p.join()

print(dict(sorted(results.items())))
```

</details>

---

## Exercise 4 - Phase synchronization with Barrier

**Background:**

A `Barrier` blocks each worker until a fixed number of participants arrive.
This is useful for staged workflows where all workers must reach the same
checkpoint before continuing together.

Demo pattern:

```python
barrier = mp.Barrier(2)

def waiter(barrier):
    print("before barrier")
    barrier.wait()
    print("after barrier")

p = dragon.native.Process(target=waiter, args=(barrier,))
p.start()
time.sleep(0.5)
barrier.wait()
p.join()
```

**Your task:**

1. Create a `Barrier` named `phase_barrier` with 6 participants (5 workers + parent)
2. Write worker code that prints `ready`, waits on the barrier, then prints `running`
3. Launch 5 workers
4. Sleep for 1 second in the parent and then call `phase_barrier.wait()`
5. Join workers

In [ ]:
# -- Exercise 4 -- your code here -----------------------------------------------


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def staged_worker(worker_id, phase_barrier):
    print(f"worker={worker_id} ready")
    phase_barrier.wait()
    print(f"worker={worker_id} running")

phase_barrier = mp.Barrier(6)
procs = [dragon.native.Process(target=staged_worker, args=(i, phase_barrier)) for i in range(5)]

for p in procs:
    p.start()

time.sleep(1.0)
print("Parent arriving at barrier")
phase_barrier.wait()

for p in procs:
    p.join()
```

</details>

---

## Exercise 5 - Multi-node process placement and reduction

**Background:**

When Dragon runs with a multi-node allocation, workers can be placed across
nodes automatically. A practical way to verify distribution is to collect each
worker's hostname and aggregate counts.

On a single-node run, all processes are expected to start on the same host.
On a multi-node run, default process placement is round-robin across all
allocated nodes. If you need tighter control, Dragon Policies can be used to
specify explicit process placement on nodes.

Demo pattern:

```python
q = mp.Queue()

def one_host_report(q):
    q.put(socket.gethostname())

p = dragon.native.Process(target=one_host_report, args=(q,))
p.start()
host = q.get()
p.join()
print("worker host:", host)
```

**Your task:**

1. Write `host_worker(i, q)` that sends `(hostname, i)` to a queue
2. Launch 24 workers
3. Collect all results and count how many workers ran on each hostname
4. Print the per-host counts
5. Compute and print the sum of worker ids (reduction check)

If your session has only one node, all workers may report the same hostname.
With a multi-node Dragon launch, you should see multiple hostnames.

In [ ]:
# -- Exercise 5 -- your code here -----------------------------------------------


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def host_worker(i, q):
    q.put((socket.gethostname(), i))

q = mp.Queue()
n_workers = 24
procs = [dragon.native.Process(target=host_worker, args=(i, q)) for i in range(n_workers)]

for p in procs:
    p.start()

host_counts = {}
id_sum = 0
for _ in range(n_workers):
    host, worker_id = q.get()
    host_counts[host] = host_counts.get(host, 0) + 1
    id_sum += worker_id

for p in procs:
    p.join()

print("Workers per host:")
for host, count in sorted(host_counts.items()):
    print(f"  {host}: {count}")

print("Reduction check (sum of ids):", id_sum)
print("Expected:", n_workers * (n_workers - 1) // 2)
```

</details>

---

## Summary

You used DragonHPC with familiar multiprocessing concepts to launch workers,
synchronize phases, communicate data, and run data-parallel map operations.
The same API patterns apply whether you run on one node or many.

| Concept | API |
|---|---|
| Start Dragon backend | `mp.set_start_method("dragon")` |
| Launch process | `dragon.native.Process(target=..., args=...)` |
| Wait for completion | `p.join()` |
| Mutual exclusion | `mp.Lock()` |
| Shared scalar | `mp.Value(...)` |
| Message passing | `mp.Queue()` |
| Phase synchronization | `mp.Barrier(n)` |
| Pool map | `mp.Pool(...).map(func, data)` |
| Multi-node placement check | `socket.gethostname()` |

### What next?

Try adapting these exercises to your own application by replacing toy data with
real workloads and scaling the worker count to your full Dragon allocation.